Wojciech Kucharski 240435

Bartłomiej Art

In [2]:
!pip install pulp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.7/17.7 MB 24.9 MB/s eta 0:00:00


In [7]:
import pulp
from pulp import *
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive, fixed, interact_manual, Layout, FloatSlider, IntSlider
from ipywidgets import interact, FloatSlider, Layout

In [4]:
prob = LpProblem("Produkcja problem",LpMaximize)
sztukA=LpVariable("sztukA",0,None,LpInteger)
sztukB=LpVariable("sztukB",0,None,LpInteger)
sztukC=LpVariable("sztukC",0,None,LpInteger)


prob += 400*sztukA + 300*sztukB +200*sztukC, "Optymalizacja zysku"
prob += 0.3*sztukA + 0.5*sztukB+0.4*sztukC <= 1800, "Montaż godzin"
prob += 0.1*sztukA + 0.08*sztukB+0.04*sztukC <= 800, "Kontrola godzin"
prob += 0.06*sztukA + 0.04*sztukB+0.05*sztukC <= 700, "Pakowanie godzin"
prob.writeLP("Kontrola.lp")
prob.solve()

for v in prob.variables():
    print(v.name, "=", v.varValue)
print("Całkowity zysk = ", value(prob.objective))

sztukA = 6000.0
sztukB = 0.0
sztukC = 0.0
Całkowity zysk =  2400000.0


/usr/local/lib/python3.11/dist-packages/pulp/pulp.py:1298: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


In [8]:



Skladniki = ["CZEKOLADA", "KARMEL", "CUKIER", "ORZECHY", "OLEJ", "OPAKOWANIA"]

mars = {
    "CZEKOLADA": 0.30,
    "KARMEL": 0.40,
    "CUKIER": 0.33,
    "ORZECHY": 0.07,
    "OLEJ": 0.06,
    "OPAKOWANIA": 1
}

snickers = {
    "CZEKOLADA": 0.45,
    "KARMEL": 0.11,
    "CUKIER": 0.22,
    "ORZECHY": 0.51,
    "OLEJ": 0.07,
    "OPAKOWANIA": 1
}

Cena = {
    "CZEKOLADA": 22,
    "KARMEL": 16,
    "ORZECHY": 21,
    "CUKIER": 11,
    "OLEJ": 8,
    "OPAKOWANIA": 2
}

styl = {'description_width': 'initial'}
czekoladaMax_suwak = FloatSlider(min=0, max=15000, value=8000,
                                  description="ograniczenieCzekolady",
                                  style=styl, layout=Layout(width='400px'))

orzechyMax_suwak = FloatSlider(min=0, max=4000, value=2000,
                                description="ograniczenieOrzechów",
                                style=styl, layout=Layout(width='400px'))

opakowaniaMax_suwak = FloatSlider(min=0, max=25000, value=12000,
                                   description="ograniczenieOpakowań",
                                   style=styl, layout=Layout(width='400px'))

marsCena_suwak = FloatSlider(min=85, max=115, value=95,
                              description="cenaMarsów",
                              style=styl, layout=Layout(width='400px'))

snickersCena_suwak = FloatSlider(min=85, max=115, value=110,
                                  description="cenaSnickersów",
                                  style=styl, layout=Layout(width='400px'))

def optymalizujProdukcje(czekoladaMax=8000, orzechyMax=2000, opakowaniaMax=12000,
                          marsCena=95, snickersCena=110):

    model = LpProblem("Problem finansowy", LpMaximize)
    marsIlosc = LpVariable('marsIlosc', lowBound=0, cat='LPInteger')
    snickersIlosc = LpVariable('snickersIlosc', lowBound=0, cat='LPInteger')
    marsKoszt, snickersKoszt = 0, 0

    for skladnik in Skladniki:
        marsKoszt += mars[skladnik] * Cena[skladnik]
        snickersKoszt += snickers[skladnik] * Cena[skladnik]

    model += ((marsCena - marsKoszt) * marsIlosc
              + (snickersCena - snickersKoszt) * snickersIlosc), "Maksymalizacja_Zysku"
    model += marsIlosc * mars["ORZECHY"] + snickersIlosc * snickers["ORZECHY"] <= orzechyMax, "orzechyMax"
    model += marsIlosc * mars["CZEKOLADA"] + snickersIlosc * snickers["CZEKOLADA"] <= czekoladaMax, "czekoladaMax"
    model += marsIlosc * mars["OPAKOWANIA"] + snickersIlosc * snickers["OPAKOWANIA"] <= opakowaniaMax, "opakowaniaMax"

    model.solve()

    print("=== Zysk na baton ===")
    print("Zysk Marsa na baton:", marsCena - marsKoszt)
    print("Zysk Snickersa na baton:", snickersCena - snickersKoszt)
    print("")

    print("=== Pozostałe zasoby ===")
    pozostala_czekolada = czekoladaMax - (marsIlosc.varValue * mars["CZEKOLADA"]
                                          + snickersIlosc.varValue * snickers["CZEKOLADA"])
    pozostale_orzechy = orzechyMax - (marsIlosc.varValue * mars["ORZECHY"]
                                      + snickersIlosc.varValue * snickers["ORZECHY"])
    pozostale_opakowania = opakowaniaMax - (marsIlosc.varValue * mars["OPAKOWANIA"]
                                            + snickersIlosc.varValue * snickers["OPAKOWANIA"])

    print("Pozostała czekolada:", pozostala_czekolada)
    print("Pozostałe orzechy:", pozostale_orzechy)
    print("Pozostałe opakowania:", pozostale_opakowania)
    print("")

    print("=== Status rozwiązania ===")
    print("Status solvera:", LpStatus[model.status])
    print("")

    print("=== Zmienne decyzyjne ===")
    for zmienna in model.variables():
        print(f"{zmienna.name} =", zmienna.varValue)
    print("")

    finalny_zysk = value(model.objective)
    print("=== Całkowity zysk ===")
    print("Całkowity zysk:", finalny_zysk)

    fig, ax = plt.subplots()
    ax.bar(["Mars", "Snickers"], [marsIlosc.varValue, snickersIlosc.varValue],
           color=["#FFA07A", "#8A2BE2"])
    ax.set_title("Optymalny Plan Produkcji")
    ax.set_ylabel("Liczba wyprodukowanych sztuk")
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

interact(
    optymalizujProdukcje,
    czekoladaMax=czekoladaMax_suwak,
    orzechyMax=orzechyMax_suwak,
    opakowaniaMax=opakowaniaMax_suwak,
    marsCena=marsCena_suwak,
    snickersCena=snickersCena_suwak
)


interactive(children=(FloatSlider(value=8000.0, description='ograniczenieCzekolady', layout=Layout(width='400p…

<function __main__.optymalizujProdukcje(czekoladaMax=8000, orzechyMax=2000, opakowaniaMax=12000, marsCena=95, snickersCena=110)>